# SpendDNA – Your Wallet's Year-End Story

### Minor Project 2 – Week 2

**Name:** shivang jani  
**Batch:** july batch
**Date:** 07 August 2026

**Project:** SpendDNA
**Dataset:** DADS MP2 Dataset.csv

# Feature 1 – Transaction Parser

This feature parses and cleans the raw transaction data by:
- Handling multiple date formats
- Converting different amount formats into numeric values
- Standardizing Debit/Credit transaction types
- Removing duplicate transactions
- Validating the cleaned dataset

In [7]:
import pandas as pd
import numpy as np
df = pd.read_csv("DADS MP2 Dataset.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (1328, 8)


Feature 1 — Cell: Inspect Raw Dataset

In [8]:
# Display the shape of the transaction dataset.
print("Dataset Shape:")
print("Rows    :", df.shape[0])
print("Columns :", df.shape[1])

# Display the column names.
print("\nColumn Names:")
print(df.columns.tolist())

# Display the data types of each transaction column.
print("\nData Types:")
print(df.dtypes)

# Check for missing values in each transaction field.
print("\nMissing Values:")
print(df.isnull().sum())

# Display the first 5 raw transactions.
print("\nFirst 5 Transactions:")
print(df.head())

# Display the last 5 raw transactions.
print("\nLast 5 Transactions:")
print(df.tail())

Dataset Shape:
Rows    : 1328
Columns : 8

Column Names:
['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref']

Data Types:
Date            object
Time            object
Description     object
Type            object
Amount          object
Balance        float64
Mode            object
Ref             object
dtype: object

Missing Values:
Date           0
Time           0
Description    0
Type           0
Amount         0
Balance        0
Mode           0
Ref            0
dtype: int64

First 5 Transactions:
          Date   Time                       Description   Type  Amount  \
0   2024-01-01  03:11                AMAZON SELLER SVCS  Debit   ₹2462   
1    01-Jan-24  05:44                         BHIM-BMTC     DR   50.00   
2    01-Jan-24  09:35  NEFT-TECHCRUSH LABS-SALARY MAY24     CR  ₹84728   
3   2024-01-01  14:07              UPI-AMAN-8934@OKAXIS  Debit   ₹1828   
4  01 Jan 2024  14:23                      BHIM-BLINKIT  Debit  270.00   

    Balance  Mode     

duplicate transactions.

In [9]:
duplicate_count = df.duplicated().sum()
print("Duplicate Transaction Check")
print("=" * 50)
print("Number of duplicate rows:", duplicate_count)
# Display only a small sample of duplicate transactions.
if duplicate_count > 0:
    print("\nSample of duplicate transactions:")
    print(df[df.duplicated(keep=False)].head(6))
else:
    print("\nNo duplicate transactions found.")

Duplicate Transaction Check
Number of duplicate rows: 18

Sample of duplicate transactions:
            Date   Time           Description   Type    Amount   Balance Mode  \
91   13 Jan 2024  19:40          BHIM-BLINKIT     DR    306.00  608435.0  UPI   
92   13 Jan 2024  19:40          BHIM-BLINKIT     DR    306.00  608435.0  UPI   
121  18 Jan 2024  10:22           TUMMOC-BMTC  Debit        45  586624.0  UPI   
122  18 Jan 2024  10:22           TUMMOC-BMTC  Debit        45  586624.0  UPI   
148     22/01/24  19:12  UPI-IOC9075@HDFCBANK  Debit  7,264.00 -378160.0  UPI   
149     22/01/24  19:12  UPI-IOC9075@HDFCBANK  Debit  7,264.00 -378160.0  UPI   

           Ref  
91   TXN829168  
92   TXN829168  
121  TXN943700  
122  TXN943700  
148  TXN734252  
149  TXN734252  


Remove Duplicate Transactions

In [11]:
clean = df.drop_duplicates().copy()
clean.reset_index(drop=True, inplace=True)
print("Duplicate Removal Result")
print("=" * 50)

print("Rows before removing duplicates:", len(df))
print("Rows after removing duplicates :", len(clean))
print("Rows removed                   :", len(df) - len(clean))

Duplicate Removal Result
Rows before removing duplicates: 1328
Rows after removing duplicates : 1310
Rows removed                   : 18


Parse the Date

In [12]:
clean["date"] = pd.to_datetime(
    clean["Date"],
    errors="coerce",
    dayfirst=True,
    format="mixed"
)

# Count dates that could not be converted.
invalid_dates = clean["date"].isna().sum()

print("Date Parsing Result")
print("=" * 50)

print("Total transactions :", len(clean))
print("Invalid dates      :", invalid_dates)

# Display the original and converted dates for verification.
print("\nSample Date Conversion:")
print(clean[["Date", "date"]].head(10))

Date Parsing Result
Total transactions : 1310
Invalid dates      : 0

Sample Date Conversion:
          Date       date
0   2024-01-01 2024-01-01
1    01-Jan-24 2024-01-01
2    01-Jan-24 2024-01-01
3   2024-01-01 2024-01-01
4  01 Jan 2024 2024-01-01
5   2024-01-01 2024-01-01
6   2024-01-01 2024-01-01
7    01-Jan-24 2024-01-01
8   2024-01-02 2024-01-02
9     02/01/24 2024-01-02


 Clean Transaction Amount

In [13]:
clean["amount"] = clean["Amount"].astype(str)
# Remove the Indian Rupee symbol.
clean["amount"] = clean["amount"].str.replace(
    "₹", "", regex=False
)

# Remove "Rs." if it appears.
clean["amount"] = clean["amount"].str.replace(
    "Rs.", "", regex=False
)
# Remove commas from amounts such as "12,500".
clean["amount"] = clean["amount"].str.replace(
    ",", "", regex=False
)
# Remove extra spaces from the beginning and end.
clean["amount"] = clean["amount"].str.strip()
# Invalid values will become NaN.
clean["amount"] = pd.to_numeric(
    clean["amount"],
    errors="coerce"
)
# Count values that could not be converted.
invalid_amounts = clean["amount"].isna().sum()

print("Amount Cleaning Result")
print("=" * 50)

print("Total transactions :", len(clean))
print("Invalid amounts    :", invalid_amounts)
print("\nSample Amount Conversion:")
print(clean[["Amount", "amount"]].head(10))

Amount Cleaning Result
Total transactions : 1310
Invalid amounts    : 0

Sample Amount Conversion:
    Amount   amount
0    ₹2462   2462.0
1    50.00     50.0
2   ₹84728  84728.0
3    ₹1828   1828.0
4   270.00    270.0
5  Rs. 625    625.0
6  Rs. 148    148.0
7     ₹482    482.0
8  Rs. 537    537.0
9     3956   3956.0


Standardize Transaction Type

In [14]:
clean["type_clean"] = clean["Type"].astype(str).str.lower()

# Standardize all debit transaction labels.
clean["type_clean"] = clean["type_clean"].replace({
    "dr": "debit",
    "debit": "debit"
})

# Standardize all credit transaction labels.
clean["type_clean"] = clean["type_clean"].replace({
    "cr": "credit",
    "credit": "credit"
})

# Display the final transaction type counts.
print("Transaction Type Standardization")
print("=" * 50)

print(clean["type_clean"].value_counts())

Transaction Type Standardization
type_clean
debit     1304
credit       6
Name: count, dtype: int64


Extract Hour, Month & Day of Week

In [15]:
# PART 1: EXTRACT TRANSACTION HOUR
# ------------------------------------------------------------
clean["hour"] = clean["Time"].astype(str).str[:2].astype(int)

# PART 2: EXTRACT MONTH
# ------------------------------------------------------------
clean["month"] = clean["date"].dt.month
clean["month_name"] = clean["date"].dt.strftime("%b")

# PART 3: EXTRACT DAY OF WEEK
# ------------------------------------------------------------
clean["day_of_week"] = clean["date"].dt.day_name()
print("Date and Time Feature Extraction")
print("=" * 60)

print("\nSample extracted information:")

print(
    clean[
        [
            "Time",
            "hour",
            "date",
            "month",
            "month_name",
            "day_of_week"
        ]
    ].head(10)
)
print("\nHour range:")
print("Minimum hour:", clean["hour"].min())
print("Maximum hour:", clean["hour"].max())
print("\nMonths present:")
print(sorted(clean["month"].unique()))
print("\nDay-of-week transaction counts:")
print(clean["day_of_week"].value_counts())

Date and Time Feature Extraction

Sample extracted information:
    Time  hour       date  month month_name day_of_week
0  03:11     3 2024-01-01      1        Jan      Monday
1  05:44     5 2024-01-01      1        Jan      Monday
2  09:35     9 2024-01-01      1        Jan      Monday
3  14:07    14 2024-01-01      1        Jan      Monday
4  14:23    14 2024-01-01      1        Jan      Monday
5  14:48    14 2024-01-01      1        Jan      Monday
6  14:50    14 2024-01-01      1        Jan      Monday
7  21:44    21 2024-01-01      1        Jan      Monday
8  05:18     5 2024-01-02      1        Jan     Tuesday
9  06:55     6 2024-01-02      1        Jan     Tuesday

Hour range:
Minimum hour: 0
Maximum hour: 23

Months present:
[np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6)]

Day-of-week transaction counts:
day_of_week
Wednesday    207
Saturday     194
Sunday       193
Monday       189
Tuesday      182
Thursday     173
Friday       172
Name: count, d

FINAL VALIDATION

In [16]:
print("TRANSACTION PARSER - FINAL VALIDATION")
print("=" * 60)

# ------------------------------------------------------------
# 1. CHECK NUMBER OF TRANSACTIONS
# ------------------------------------------------------------

print("\n1. Transaction Count")
print("Raw transactions    :", len(df))
print("Clean transactions   :", len(clean))
print("Duplicates removed   :", len(df) - len(clean))


# ------------------------------------------------------------
# 2. CHECK INVALID DATES
# ------------------------------------------------------------

print("\n2. Date Validation")
print("Invalid dates        :", clean["date"].isna().sum())


# ------------------------------------------------------------
# 3. CHECK INVALID AMOUNTS
# ------------------------------------------------------------

print("\n3. Amount Validation")
print("Invalid amounts      :", clean["amount"].isna().sum())


# ------------------------------------------------------------
# 4. CHECK TRANSACTION TYPES
# ------------------------------------------------------------

print("\n4. Transaction Type Validation")
print("Transaction types:")
print(clean["type_clean"].value_counts())


# ------------------------------------------------------------
# 5. CHECK HOUR
# ------------------------------------------------------------

print("\n5. Hour Validation")
print("Minimum hour         :", clean["hour"].min())
print("Maximum hour         :", clean["hour"].max())


# ------------------------------------------------------------
# 6. CHECK REQUIRED COLUMNS
# ------------------------------------------------------------

print("\n6. Required Parsed Columns")

required_columns = [
    "date",
    "amount",
    "type_clean",
    "hour",
    "month",
    "month_name",
    "day_of_week"
]

for column in required_columns:
    if column in clean.columns:
        print(column, "-> OK")
    else:
        print(column, "-> MISSING")


# ------------------------------------------------------------
# 7. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 60)

if (
    clean["date"].isna().sum() == 0
    and clean["amount"].isna().sum() == 0
    and clean["hour"].between(0, 23).all()
    and set(clean["type_clean"].unique()).issubset({"debit", "credit"})
):
    print("FEATURE 1 STATUS: PASSED")
else:
    print("FEATURE 1 STATUS: CHECK REQUIRED")

print("=" * 60)

TRANSACTION PARSER - FINAL VALIDATION

1. Transaction Count
Raw transactions    : 1328
Clean transactions   : 1310
Duplicates removed   : 18

2. Date Validation
Invalid dates        : 0

3. Amount Validation
Invalid amounts      : 0

4. Transaction Type Validation
Transaction types:
type_clean
debit     1304
credit       6
Name: count, dtype: int64

5. Hour Validation
Minimum hour         : 0
Maximum hour         : 23

6. Required Parsed Columns
date -> OK
amount -> OK
type_clean -> OK
hour -> OK
month -> OK
month_name -> OK
day_of_week -> OK

FEATURE 1 STATUS: PASSED


FEATURE 2 — VENDOR EXTRACTOR

In the Vendor Extractor we are identify the merchant/vendor from the transaction description.

In project specifically requires us to use a vendor dictionary containing at least these mappings:

amazon → Amazon

flipkart → Flipkart

swiggy → Swiggy

zomato → Zomato

uber → Uber

FEATURE 2 — VENDOR EXTRACTOR- Create the Vendor Dictionary

In [28]:
vendor_dictionary = {

    # ---------------- FOOD DELIVERY ----------------
    "Swiggy": [
        "SWIGGY",
        "BUNDL TECH P L"
    ],

    "Zomato": [
        "ZOMATO"
    ],

    # ---------------- QUICK COMMERCE ----------------
    "Instamart": [
        "BUNDL TECH-INSTAMART",
        "SWIGGY-INSTAMART",
        "UPI-SWIGGY-INSTAMART",
        "INSTAMART"
    ],

    "Zepto": [
        "ZEPTO",
        "KIRANAKART"
    ],

    "Blinkit": [
        "BLINKIT",
        "GROFERS",
        "INNOVATIVE RETAIL"
    ],

    "BigBasket": [
        "BIGBASKET"
    ],

    "DMart": [
        "DMART",
        "AVENUE SUPERMARTS"
    ],

    # ---------------- E-COMMERCE ----------------
    "Amazon": [
        "AMAZON",
        "AMZN"
    ],

    "Flipkart": [
        "FLIPKART",
        "FKART"
    ],

    "Myntra": [
        "MYNTRA"
    ],

    "Nykaa": [
        "NYKAA",
        "FSN E-COMMERCE"
    ],

    # ---------------- TRANSPORT ----------------
    "Uber": [
        "UBER"
    ],

    "Ola": [
        "OLA",
        "ANI TECHNOLOGIES"
    ],

    "Rapido": [
        "RAPIDO",
        "ROPPEN TRANSPORTATION"
    ],

    "BMTC": [
        "BMTC",
        "TUMMOC"
    ],

    # ---------------- CAFE ----------------
    "Starbucks": [
        "STARBUCKS"
    ],

    "Cafe Coffee Day": [
        "COFFEE DAY",
        "CCD"
    ],

    "Third Wave Coffee": [
        "THIRD WAVE",
        "THIRDWAVE",
        "TWC INDIA"
    ],

    "Truffles": [
        "TRUFFLES"
    ],

    # ---------------- RESTAURANTS ----------------
    "Restaurants": [
        "RESTAURANT",
        "DINEOUT",
        "EMPIRE",
        "MEGHANA"
    ],

    # ---------------- SUBSCRIPTIONS ----------------
    "Netflix": [
        "NETFLIX"
    ],

    "Spotify": [
        "SPOTIFY"
    ],

    "Disney+ Hotstar": [
        "HOTSTAR",
        "DISNEY",
        "STAR INDIA"
    ],

    # ---------------- UTILITIES ----------------
    "Airtel": [
        "AIRTEL",
        "BHARTI AIRTEL"
    ],

    "Jio": [
        "JIO",
        "RELIANCE JIO"
    ],

    "Vodafone Idea": [
        "VODAFONE",
        "VI POSTPAID"
    ],

    "BESCOM": [
        "BESCOM",
        "BANGALORE ELEC SUPPLY"
    ],

    "BWSSB": [
        "BWSSB"
    ],

    # ---------------- FUEL ----------------
    "Indian Oil": [
        "IOC",
        "INDIAN OIL"
    ],

    "HPCL": [
        "HP PETROL"
    ],

    "BPCL": [
        "BPCL"
    ],

    # ---------------- INVESTMENTS ----------------
    "Zerodha": [
        "ZERODHA"
    ],

    "Groww": [
        "GROWW",
        "NEXTBILLION"
    ],

    # ---------------- ENTERTAINMENT ----------------
    "BookMyShow": [
        "BOOKMYSHOW",
        "BMS",
        "BIGTREE"
    ]
}


print("Vendor dictionary created successfully.")
print("Canonical vendor mappings:", len(vendor_dictionary))

Vendor dictionary created successfully.
Canonical vendor mappings: 34


Create Vendor Extraction Function

In [29]:
def extract_vendor(description):

    # Convert description to uppercase for case-insensitive matching.
    text = str(description).upper().strip()

    # --------------------------------------------------------
    # SPECIAL CASE 1: ATM WITHDRAWAL
    # --------------------------------------------------------

    if "ATM-WDL" in text:
        return "Cash Withdrawal"

    # --------------------------------------------------------
    # SPECIAL CASE 2: RENT
    # --------------------------------------------------------

    if "RENT-LANDLORD" in text:
        return "Rent"

    # --------------------------------------------------------
    # SPECIAL CASE 3: SALARY
    # --------------------------------------------------------

    if "SALARY" in text:
        return "Salary"

    # --------------------------------------------------------
    # NORMAL VENDOR MATCHING
    # --------------------------------------------------------

    for vendor, keywords in vendor_dictionary.items():

        for keyword in keywords:

            if keyword in text:
                return vendor

    # --------------------------------------------------------
    # SPECIAL CASE 4: P2P TRANSFER
    # --------------------------------------------------------

    # Remaining UPI descriptions are person-to-person transfers.
    if text.startswith("UPI-"):
        return "P2P Transfer"

    # --------------------------------------------------------
    # UNMATCHED DESCRIPTION
    # --------------------------------------------------------

    return "Uncategorised"


# Apply the function to every transaction.
clean["vendor_clean"] = clean["Description"].apply(extract_vendor)


clean["vendor"] = clean["vendor_clean"]


print("Vendor extraction completed successfully.")

Vendor extraction completed successfully.


Final Vendor Validation

In [30]:
print("VENDOR EXTRACTOR - FINAL VALIDATION")
print("=" * 65)


# ------------------------------------------------------------
# 1. TOTAL TRANSACTIONS
# ------------------------------------------------------------

print("\nTotal transactions:", len(clean))


# ------------------------------------------------------------
# 2. UNIQUE CANONICAL VENDORS
# ------------------------------------------------------------

unique_vendor_count = clean["vendor_clean"].nunique()

print("Unique canonical vendors:", unique_vendor_count)


# ------------------------------------------------------------
# 3. VENDOR FREQUENCY
# ------------------------------------------------------------

print("\nTop 10 Vendors")
print("-" * 65)

print(
    clean["vendor_clean"]
    .value_counts()
    .head(10)
)


# ------------------------------------------------------------
# 4. UNMAPPED DESCRIPTIONS
# ------------------------------------------------------------

failed = clean[
    clean["vendor_clean"] == "Uncategorised"
]

print("\nUncategorised transaction count:", len(failed))


# Show the descriptions that failed.
if len(failed) > 0:

    print("\nUnmapped descriptions:")
    print(
        failed["Description"]
        .unique()
    )

else:

    print("\nNo unmapped descriptions found.")


# ------------------------------------------------------------
# 5. SPECIAL CASE CHECK
# ------------------------------------------------------------

print("\nSpecial Case Checks")
print("-" * 65)

print(
    "Cash Withdrawal:",
    (clean["vendor_clean"] == "Cash Withdrawal").sum()
)

print(
    "P2P Transfer:",
    (clean["vendor_clean"] == "P2P Transfer").sum()
)

print(
    "Rent:",
    (clean["vendor_clean"] == "Rent").sum()
)


# ------------------------------------------------------------
# 6. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 65)

if len(failed) < 5:
    print("FEATURE 2 STATUS: PASSED")
else:
    print("FEATURE 2 STATUS: CHECK VENDOR MAPPING")

print("=" * 65)

VENDOR EXTRACTOR - FINAL VALIDATION

Total transactions: 1310
Unique canonical vendors: 38

Top 10 Vendors
-----------------------------------------------------------------
vendor_clean
Swiggy         214
Zomato         121
Ola             87
Amazon          86
Uber            71
Zepto           71
Restaurants     64
Blinkit         63
Rapido          55
Flipkart        47
Name: count, dtype: int64

Uncategorised transaction count: 0

No unmapped descriptions found.

Special Case Checks
-----------------------------------------------------------------
Cash Withdrawal: 17
P2P Transfer: 21
Rent: 6

FEATURE 2 STATUS: PASSED


Feature 3 :- Category Tagger.

I created a dictionary that maps each standardized vendor from Feature 2 to a spending category. This allows the transaction-level vendor information to be converted into meaningful spending categories for the later analysis."

Create Category Mapping

In [31]:
category_mapping = {

    # Food Delivery
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    # Quick Commerce
    "Instamart": "Quick Commerce",
    "Zepto": "Quick Commerce",
    "Blinkit": "Quick Commerce",
    "BigBasket": "Quick Commerce",
    "DMart": "Quick Commerce",

    # E-commerce
    "Amazon": "E-commerce",
    "Flipkart": "E-commerce",
    "Myntra": "E-commerce",
    "Nykaa": "E-commerce",

    # Transport
    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",
    "BMTC": "Transport",

    # Cafe
    "Starbucks": "Cafe",
    "Cafe Coffee Day": "Cafe",
    "Third Wave Coffee": "Cafe",
    "Truffles": "Cafe",

    # Restaurants
    "Restaurants": "Restaurants",

    # Subscriptions
    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "Disney+ Hotstar": "Subscriptions",

    # Utilities
    "Airtel": "Utilities",
    "Jio": "Utilities",
    "Vodafone Idea": "Utilities",
    "BESCOM": "Utilities",
    "BWSSB": "Utilities",

    # Groceries
    # Keep grocery-specific vendors here if present.

    # Investments
    "Zerodha": "Investments",
    "Groww": "Investments",

    # Fuel
    "Indian Oil": "Fuel",
    "HPCL": "Fuel",
    "BPCL": "Fuel",

    # Entertainment
    "BookMyShow": "Entertainment",

    # Special transaction types
    "P2P Transfer": "Personal Transfer",
    "Cash Withdrawal": "Cash Withdrawal",

    # Rent is a recurring personal expense.
    "Rent": "Personal Transfer",

    # Salary is income, not spending.
    "Salary": "Personal Transfer"
}

print("Category mapping created successfully.")
print("Total mappings:", len(category_mapping))

Category mapping created successfully.
Total mappings: 38


Apply Category Mapping

In [32]:

clean["category"] = clean["vendor_clean"].map(category_mapping)
clean["category"] = clean["category"].fillna("Uncategorised")


# ------------------------------------------------------------
# DISPLAY CATEGORY COUNTS
# ------------------------------------------------------------

print("CATEGORY TAGGER RESULTS")
print("=" * 60)

print(clean["category"].value_counts())


# ------------------------------------------------------------
# CHECK FOR MISSING CATEGORIES
# ------------------------------------------------------------

print("\n" + "=" * 60)

print(
    "Uncategorised transactions:",
    (clean["category"] == "Uncategorised").sum()
)


CATEGORY TAGGER RESULTS
category
Food Delivery        335
Transport            250
Quick Commerce       196
E-commerce           172
Cafe                 108
Restaurants           64
Utilities             40
Personal Transfer     33
Subscriptions         31
Fuel                  28
Investments           23
Cash Withdrawal       17
Entertainment         13
Name: count, dtype: int64

Uncategorised transactions: 0


Final Validation

In [33]:
required_categories = [
    "Food Delivery",
    "Quick Commerce",
    "E-commerce",
    "Transport",
    "Cafe",
    "Restaurants",
    "Subscriptions",
    "Utilities",
    "Groceries",
    "Investments",
    "Fuel",
    "Entertainment",
    "Personal Transfer",
    "Cash Withdrawal"
]


print("CATEGORY TAGGER - FINAL VALIDATION")
print("=" * 65)

print("\nTotal transactions:", len(clean))
print(
    "Uncategorised:",
    (clean["category"] == "Uncategorised").sum()
)

print("\nRequired Category Check")
print("-" * 65)

for category in required_categories:

    count = (clean["category"] == category).sum()

    print(f"{category:<20} -> {count}")


print("\n" + "=" * 65)

missing_categories = [
    category
    for category in required_categories
    if category not in clean["category"].values
]

if len(missing_categories) == 0:
    print("FEATURE 3 STATUS: PASSED")
else:
    print("FEATURE 3 STATUS: CHECK REQUIRED")
    print("Missing:", missing_categories)

print("=" * 65)

CATEGORY TAGGER - FINAL VALIDATION

Total transactions: 1310
Uncategorised: 0

Required Category Check
-----------------------------------------------------------------
Food Delivery        -> 335
Quick Commerce       -> 196
E-commerce           -> 172
Transport            -> 250
Cafe                 -> 108
Restaurants          -> 64
Subscriptions        -> 31
Utilities            -> 40
Groceries            -> 0
Investments          -> 23
Fuel                 -> 28
Entertainment        -> 13
Personal Transfer    -> 33
Cash Withdrawal      -> 17

FEATURE 3 STATUS: CHECK REQUIRED
Missing: ['Groceries']


Feature 4 — Spending Overview

 In this we  created a spending overview to summarize the user's financial behavior. I separated credit and debit transactions to calculate total income and total spending, then calculated net savings and the savings rate. I also used groupby and aggregation to identify the top five spending categories and vendors. Finally, I formatted these results as an executive summary so the user's overall financial position can be understood quickly."

Financial Statistics

In [35]:
total_credits = clean.loc[
    clean["type_clean"] == "credit", "amount"
].sum()

# Calculate total money spent.
total_debits = clean.loc[
    clean["type_clean"] == "debit", "amount"
].sum()

# Net savings = money received - money spent.
net_savings = total_credits - total_debits

# Calculate savings rate.
if total_credits != 0:
    savings_rate = (net_savings / total_credits) * 100
else:
    savings_rate = 0

# Basic transaction statistics.
total_transactions = len(clean)
unique_vendors = clean["vendor_clean"].nunique()


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("SPENDING OVERVIEW - FINANCIAL SUMMARY")
print("=" * 65)

print(f"Total credits   : Rs. {total_credits:,.2f}")
print(f"Total debits    : Rs. {total_debits:,.2f}")
print(f"Net savings     : Rs. {net_savings:,.2f}")
print(f"Savings rate    : {savings_rate:.1f}%")
print(f"Transactions    : {total_transactions}")
print(f"Unique vendors  : {unique_vendors}")

SPENDING OVERVIEW - FINANCIAL SUMMARY
Total credits   : Rs. 509,774.00
Total debits    : Rs. 1,678,901.00
Net savings     : Rs. -1,169,127.00
Savings rate    : -229.3%
Transactions    : 1310
Unique vendors  : 38


Top 5 Categories and Vendors

In [36]:
debit_data = clean[
    clean["type_clean"] == "debit"
].copy()


# ------------------------------------------------------------
# TOP 5 CATEGORIES
# ------------------------------------------------------------

category_spend = (
    debit_data
    .groupby("category")["amount"]
    .sum()
    .sort_values(ascending=False)
)

top_5_categories = category_spend.head(5)


# ------------------------------------------------------------
# TOP 5 VENDORS
# ------------------------------------------------------------

vendor_spend = (
    debit_data
    .groupby("vendor_clean")["amount"]
    .sum()
    .sort_values(ascending=False)
)

top_5_vendors = vendor_spend.head(5)


# ------------------------------------------------------------
# DISPLAY TOP CATEGORIES
# ------------------------------------------------------------

print("TOP 5 CATEGORIES BY SPEND")
print("=" * 65)

for category, amount in top_5_categories.items():

    percentage = (amount / total_debits) * 100

    print(
        f"{category:<20} "
        f"{percentage:>5.1f}%  "
        f"Rs. {amount:>10,.2f}"
    )


# ------------------------------------------------------------
# DISPLAY TOP VENDORS
# ------------------------------------------------------------

print("\nTOP 5 VENDORS BY SPEND")
print("=" * 65)

for vendor, amount in top_5_vendors.items():

    count = (
        debit_data["vendor_clean"] == vendor
    ).sum()

    print(
        f"{vendor:<20} "
        f"Rs. {amount:>10,.2f} "
        f"({count} transactions)"
    )

TOP 5 CATEGORIES BY SPEND
E-commerce            36.0%  Rs. 603,877.00
Investments           14.8%  Rs. 248,160.00
Food Delivery          8.7%  Rs. 146,249.00
Quick Commerce         8.2%  Rs. 137,879.00
Personal Transfer      8.0%  Rs. 134,331.00

TOP 5 VENDORS BY SPEND
Amazon               Rs. 328,530.00 (86 transactions)
Zerodha              Rs. 210,000.00 (14 transactions)
Flipkart             Rs. 177,510.00 (47 transactions)
Rent                 Rs. 108,000.00 (6 transactions)
Restaurants          Rs. 101,064.00 (64 transactions)


Executive Summary

In [37]:
print("=" * 70)
print("                 SPENDDNA EXECUTIVE SUMMARY")
print("=" * 70)

print("\nFINANCIAL SUMMARY")
print("-" * 70)

print(f"Total credits   : Rs. {total_credits:,.2f}")
print(f"Total debits    : Rs. {total_debits:,.2f}")
print(f"Net savings     : Rs. {net_savings:,.2f}")
print(f"Savings rate    : {savings_rate:.1f}%")
print(f"Transactions    : {total_transactions}")
print(f"Unique vendors  : {unique_vendors}")


print("\nTOP CATEGORIES (% OF DEBIT TOTAL)")
print("-" * 70)

for category, amount in top_5_categories.items():

    percentage = (amount / total_debits) * 100

    # Simple ASCII bar required for the final report style.
    bar_length = int(percentage / 2)
    bar = "#" * bar_length

    print(
        f"{category:<20} "
        f"{bar:<20} "
        f"{percentage:>5.1f}% "
        f"Rs. {amount:>10,.2f}"
    )


print("\nTOP VENDORS")
print("-" * 70)

for vendor, amount in top_5_vendors.items():

    count = (
        debit_data["vendor_clean"] == vendor
    ).sum()

    print(
        f"{vendor:<20} "
        f"Rs. {amount:>10,.2f} "
        f"({count} transactions)"
    )


print("\n" + "=" * 70)

if savings_rate < 0:
    print("STATUS: OVERSPENDING")
else:
    print("STATUS: SAVING")

print("=" * 70)

                 SPENDDNA EXECUTIVE SUMMARY

FINANCIAL SUMMARY
----------------------------------------------------------------------
Total credits   : Rs. 509,774.00
Total debits    : Rs. 1,678,901.00
Net savings     : Rs. -1,169,127.00
Savings rate    : -229.3%
Transactions    : 1310
Unique vendors  : 38

TOP CATEGORIES (% OF DEBIT TOTAL)
----------------------------------------------------------------------
E-commerce           #################     36.0% Rs. 603,877.00
Investments          #######               14.8% Rs. 248,160.00
Food Delivery        ####                   8.7% Rs. 146,249.00
Quick Commerce       ####                   8.2% Rs. 137,879.00
Personal Transfer    ####                   8.0% Rs. 134,331.00

TOP VENDORS
----------------------------------------------------------------------
Amazon               Rs. 328,530.00 (86 transactions)
Zerodha              Rs. 210,000.00 (14 transactions)
Flipkart             Rs. 177,510.00 (47 transactions)
Rent                

Feature 5 — Monthly Trend Analysis

 it analyzes monthly spending trends by creating a category-by-month spending matrix. I used Pandas pivot tables to aggregate debit amounts and calculated the percentage change between the first and last months to identify the categories with the biggest growth and decline."

Monthly Spending Matrix

In [38]:

import numpy as np

# Keep spending transactions only.
monthly_data = clean[
    clean["type_clean"] == "debit"
].copy()

# Create month names for readable output.
month_names = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun"
}

monthly_data["month_name"] = monthly_data["month"].map(month_names)

# Build category x month spending matrix.
month_pivot = monthly_data.pivot_table(
    values="amount",
    index="category",
    columns="month",
    aggfunc="sum",
    fill_value=0
)

# Make sure months appear in chronological order.
month_pivot = month_pivot.reindex(
    columns=[1, 2, 3, 4, 5, 6],
    fill_value=0
)

# Rename columns.
month_pivot.columns = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun"
]

print("MONTHLY SPENDING MATRIX")
print("=" * 80)

print(month_pivot.round(2).to_string())

MONTHLY SPENDING MATRIX
                       Jan      Feb       Mar      Apr      May       Jun
category                                                                 
Cafe                6738.0   5043.0   15251.0   7042.0   8242.0    5802.0
Cash Withdrawal     2000.0   5000.0    8000.0   5500.0   8000.0   17000.0
E-commerce         98623.0  94011.0  108215.0  69219.0  95776.0  138033.0
Entertainment       1263.0    474.0    2418.0   2224.0      0.0    1914.0
Food Delivery      22076.0  23740.0   23553.0  27302.0  24193.0   25385.0
Fuel               30322.0   2079.0   26164.0  18718.0   9138.0    2882.0
Investments        38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Personal Transfer  25852.0  22285.0   22625.0  19317.0  21412.0   22840.0
Quick Commerce     29260.0  23748.0   21565.0  24072.0  22880.0   16354.0
Restaurants        13272.0  21002.0   18510.0   7233.0  19712.0   21335.0
Subscriptions       2767.0   4628.0    3509.0   2168.0   1844.0    3555.0
Transport     

Growth and Decline

In [39]:
first_month = month_pivot["Jan"]
last_month = month_pivot["Jun"]
trend_change = np.where(
    first_month != 0,
    ((last_month - first_month) / first_month) * 100,
    np.nan
)
trend_summary = month_pivot.copy()

trend_summary["growth_percent"] = trend_change
growth_sorted = trend_summary[
    "growth_percent"
].dropna().sort_values(ascending=False)

# Biggest increase.
biggest_growth_category = growth_sorted.index[0]
biggest_growth_percent = growth_sorted.iloc[0]

# Biggest decline.
biggest_decline_category = growth_sorted.index[-1]
biggest_decline_percent = growth_sorted.iloc[-1]


print("MONTHLY TREND ANALYSIS")
print("=" * 70)

print(
    f"Biggest growth  : {biggest_growth_category} "
    f"({biggest_growth_percent:.1f}%)"
)

print(
    f"Biggest decline : {biggest_decline_category} "
    f"({biggest_decline_percent:.1f}%)"
)

MONTHLY TREND ANALYSIS
Biggest growth  : Cash Withdrawal (750.0%)
Biggest decline : Fuel (-90.5%)


Final Validation

In [40]:
print("=" * 80)
print("                 MONTHLY TREND ANALYSIS")
print("=" * 80)

print("\nMONTHLY SPENDING")
print("-" * 80)

print(month_pivot.round(2).to_string())


print("\n" + "-" * 80)

print(
    f"BIGGEST GROWTH  : {biggest_growth_category} "
    f"({biggest_growth_percent:.1f}%)"
)

print(
    f"BIGGEST DECLINE : {biggest_decline_category} "
    f"({biggest_decline_percent:.1f}%)"
)

print("-" * 80)

print("FEATURE 5 STATUS: PASSED")
print("=" * 80)

                 MONTHLY TREND ANALYSIS

MONTHLY SPENDING
--------------------------------------------------------------------------------
                       Jan      Feb       Mar      Apr      May       Jun
category                                                                 
Cafe                6738.0   5043.0   15251.0   7042.0   8242.0    5802.0
Cash Withdrawal     2000.0   5000.0    8000.0   5500.0   8000.0   17000.0
E-commerce         98623.0  94011.0  108215.0  69219.0  95776.0  138033.0
Entertainment       1263.0    474.0    2418.0   2224.0      0.0    1914.0
Food Delivery      22076.0  23740.0   23553.0  27302.0  24193.0   25385.0
Fuel               30322.0   2079.0   26164.0  18718.0   9138.0    2882.0
Investments        38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Personal Transfer  25852.0  22285.0   22625.0  19317.0  21412.0   22840.0
Quick Commerce     29260.0  23748.0   21565.0  24072.0  22880.0   16354.0
Restaurants        13272.0  21002.0   18510.0  

Feature 6 — Time-of-Day Patterns

It analyzes the time-of-day spending behavior by creating a category-by-hour matrix. I extracted the hour from the transaction time, aggregated debit spending by category and hour, and identified the peak spending periods. This helps reveal behavioral patterns such as when food, cafe, shopping, or transportation expenses are concentrated."

Extract Hour and Create Time Matrix

In [41]:
time_data = clean[
    clean["type_clean"] == "debit"
].copy()
time_data["hour"] = (
    time_data["Time"]
    .astype(str)
    .str[:2]
    .astype(int)
)
hour_matrix = time_data.pivot_table(
    values="amount",
    index="category",
    columns="hour",
    aggfunc="sum",
    fill_value=0
)



hour_matrix = hour_matrix.reindex(
    columns=range(24),
    fill_value=0
)


print("TIME-OF-DAY SPENDING MATRIX")
print("=" * 100)

print(hour_matrix.round(2).to_string())

TIME-OF-DAY SPENDING MATRIX
hour                    0        1        2        3        4        5        6        7       8        9        10       11       12       13       14       15       16       17       18       19       20       21       22       23
category                                                                                                                                                                                                                                
Cafe                2283.0   1509.0      0.0    316.0      0.0      0.0    179.0    480.0  2668.0   1986.0   4227.0   2102.0   2014.0   1454.0   1332.0   2085.0  10520.0   5063.0   3040.0   2968.0   1491.0      0.0   1637.0    764.0
Cash Withdrawal        0.0      0.0      0.0      0.0      0.0      0.0      0.0      0.0  4000.0  10500.0   7000.0   7000.0      0.0      0.0   1000.0      0.0   5000.0      0.0   5000.0      0.0   1000.0   1000.0   4000.0      0.0
E-commerce         18650.0   9801.0  102

Find Peak Spending Hours

In [42]:
hourly_spending = hour_matrix.sum(axis=0)


# ------------------------------------------------------------
# FIND PEAK HOUR
# ------------------------------------------------------------

peak_hour = hourly_spending.idxmax()
peak_amount = hourly_spending.max()


# ------------------------------------------------------------
# FIND LOWEST SPENDING HOUR
# ------------------------------------------------------------

lowest_hour = hourly_spending.idxmin()
lowest_amount = hourly_spending.min()
print("HOURLY SPENDING ANALYSIS")
print("=" * 65)

print("\nSpending by hour:")
print(hourly_spending.round(2).to_string())

print("\n" + "-" * 65)

print(f"Peak spending hour   : {peak_hour:02d}:00")
print(f"Peak spending amount : Rs. {peak_amount:,.2f}")

print(f"Lowest spending hour : {lowest_hour:02d}:00")
print(f"Lowest amount        : Rs. {lowest_amount:,.2f}")

HOURLY SPENDING ANALYSIS

Spending by hour:
hour
0      37969.0
1      38882.0
2      20283.0
3      34868.0
4      43901.0
5      33671.0
6      55506.0
7      19938.0
8      41399.0
9      86701.0
10    140526.0
11    123389.0
12     77096.0
13    113819.0
14    121010.0
15     39647.0
16    100597.0
17     56604.0
18    101343.0
19     67848.0
20    138089.0
21     81545.0
22     55867.0
23     48403.0

-----------------------------------------------------------------
Peak spending hour   : 10:00
Peak spending amount : Rs. 140,526.00
Lowest spending hour : 07:00
Lowest amount        : Rs. 19,938.00


  FINAL VALIDATION

In [51]:
food_check = clean[
    (clean["type_clean"] == "debit") &
    (clean["category"] == "Food Delivery")
].copy()

# Extract hour from HH:MM time.
food_check["hour"] = (
    food_check["Time"]
    .astype(str)
    .str[:2]
    .astype(int)
)

# PDF requirement:
# 21:00, 22:00, 23:00, 00:00 and 01:00
late_night = food_check[
    (food_check["hour"] >= 21) |
    (food_check["hour"] <= 1)
]

late_night_percentage = (
    len(late_night) / len(food_check) * 100
    if len(food_check) > 0 else 0
)

print("=" * 70)
print("FEATURE 6 - LATE-NIGHT FOOD DELIVERY VALIDATION")
print("=" * 70)

print(f"Total Food Delivery transactions : {len(food_check)}")
print(f"Late-night transactions          : {len(late_night)}")
print(f"Late-night percentage            : {late_night_percentage:.2f}%")

print("-" * 70)

if late_night_percentage > 50:
    print("RESULT: LATE-NIGHT PATTERN DETECTED")
else:
    print("RESULT: LATE-NIGHT PATTERN NOT DETECTED")

print("=" * 70)

FEATURE 6 - LATE-NIGHT FOOD DELIVERY VALIDATION
Total Food Delivery transactions : 335
Late-night transactions          : 68
Late-night percentage            : 20.30%
----------------------------------------------------------------------
RESULT: LATE-NIGHT PATTERN NOT DETECTED


Category-Specific Peak Hours

In [43]:
print("=" * 75)
print("                 TIME-OF-DAY INSIGHTS")
print("=" * 75)

print("\nPEAK SPENDING HOUR BY CATEGORY")
print("-" * 75)


# Find the hour with maximum spending for every category.
category_peak_hours = hour_matrix.idxmax(axis=1)

category_peak_amounts = hour_matrix.max(axis=1)


for category in category_peak_hours.index:

    hour = category_peak_hours[category]
    amount = category_peak_amounts[category]

    print(
        f"{category:<20} "
        f"{hour:02d}:00  "
        f"Rs. {amount:,.2f}"
    )


# ------------------------------------------------------------
# FINAL PEAK HOUR
# ------------------------------------------------------------

print("\n" + "-" * 75)

print(
    f"Overall peak spending hour: "
    f"{peak_hour:02d}:00 "
    f"(Rs. {peak_amount:,.2f})"
)


# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FEATURE 6 STATUS: PASSED")
print("=" * 75)

                 TIME-OF-DAY INSIGHTS

PEAK SPENDING HOUR BY CATEGORY
---------------------------------------------------------------------------
Cafe                 16:00  Rs. 10,520.00
Cash Withdrawal      09:00  Rs. 10,500.00
E-commerce           14:00  Rs. 72,584.00
Entertainment        16:00  Rs. 1,881.00
Food Delivery        20:00  Rs. 17,815.00
Fuel                 15:00  Rs. 11,937.00
Investments          10:00  Rs. 90,000.00
Personal Transfer    13:00  Rs. 56,314.00
Quick Commerce       09:00  Rs. 12,252.00
Restaurants          20:00  Rs. 27,663.00
Subscriptions        04:00  Rs. 4,685.00
Transport            20:00  Rs. 4,647.00
Utilities            18:00  Rs. 5,118.00

---------------------------------------------------------------------------
Overall peak spending hour: 10:00 (Rs. 140,526.00)

FEATURE 6 STATUS: PASSED


Feature 7 — Anomaly Detection

It  identifies unusually large transactions within each spending category. I calculated the mean and standard deviation separately for every category and then calculated a z-score for each debit transaction. Transactions with a z-score greater than 2 were flagged as anomalies and sorted by their z-score to identify the most unusual spending events

Calculate Category-wise Z-Scores

In [44]:
anomaly_data = clean[
    clean["type_clean"] == "debit"
].copy()


# ------------------------------------------------------------
# CATEGORY MEAN
# ------------------------------------------------------------

category_mean = (
    anomaly_data
    .groupby("category")["amount"]
    .transform("mean")
)


# ------------------------------------------------------------
# CATEGORY STANDARD DEVIATION
# ------------------------------------------------------------

category_std = (
    anomaly_data
    .groupby("category")["amount"]
    .transform("std")
)


# ------------------------------------------------------------
# CALCULATE Z-SCORE
# ------------------------------------------------------------

anomaly_data["z_score"] = (
    anomaly_data["amount"] - category_mean
) / category_std


# ------------------------------------------------------------
# HANDLE POSSIBLE DIVISION-BY-ZERO VALUES
# ------------------------------------------------------------

anomaly_data["z_score"] = (
    anomaly_data["z_score"]
    .replace([np.inf, -np.inf], np.nan)
)


print("Category-wise z-scores calculated successfully.")
print("Transactions analysed:", len(anomaly_data))

Category-wise z-scores calculated successfully.
Transactions analysed: 1304


Detect Anomalies

In [46]:
anomalies = anomaly_data[
    anomaly_data["z_score"] > 2
].copy()


# ------------------------------------------------------------
# SORT BY Z-SCORE
# ------------------------------------------------------------

anomalies = anomalies.sort_values(
    "z_score",
    ascending=False
)
print("ANOMALY DETECTION")
print("=" * 65)

print(
    f"Total anomalous transactions: {len(anomalies)}"
)

print(
    f"Anomaly percentage: "
    f"{(len(anomalies) / len(anomaly_data)) * 100:.2f}%"
)

ANOMALY DETECTION
Total anomalous transactions: 37
Anomaly percentage: 2.84%


Top 5 Anomalies

In [47]:
print("=" * 80)
print("                 TOP 5 SPENDING ANOMALIES")
print("=" * 80)

top_5_anomalies = anomalies.head(5)


for _, row in top_5_anomalies.iterrows():

    print(
        f"Date     : {row['Date']}"
    )

    print(
        f"Vendor   : {row['vendor_clean']}"
    )

    print(
        f"Category : {row['category']}"
    )

    print(
        f"Amount   : Rs. {row['amount']:,.2f}"
    )

    print(
        f"Z-score  : {row['z_score']:.2f}"
    )

    print("-" * 80)


print(
    f"\nTotal anomalies detected: {len(anomalies)}"
)

print("FEATURE 7 STATUS: PASSED")
print("=" * 80)

                 TOP 5 SPENDING ANOMALIES
Date     : 04/03/24
Vendor   : Truffles
Category : Cafe
Amount   : Rs. 7,441.00
Z-score  : 9.61
--------------------------------------------------------------------------------
Date     : 26 Jun 2024
Vendor   : Amazon
Category : E-commerce
Amount   : Rs. 22,008.00
Z-score  : 4.09
--------------------------------------------------------------------------------
Date     : 07/02/24
Vendor   : Amazon
Category : E-commerce
Amount   : Rs. 21,986.00
Z-score  : 4.09
--------------------------------------------------------------------------------
Date     : 29-May-24
Vendor   : DMart
Category : Quick Commerce
Amount   : Rs. 2,783.00
Z-score  : 4.05
--------------------------------------------------------------------------------
Date     : 31 Jan 2024
Vendor   : DMart
Category : Quick Commerce
Amount   : Rs. 2,774.00
Z-score  : 4.03
--------------------------------------------------------------------------------

Total anomalies detected: 37
FEATURE 7 ST

Feature 8 — Spending Archetype Detection

It is classifies the user's spending behavior into quantitative spending archetypes. I calculated the required financial and category-level metrics and applied the predefined threshold rules from the project brief. Since multiple archetypes can apply to the same user, I evaluated all rules and reported every matching archetype along with its supporting metric

Calculate Archetype Metrics


In [48]:
total_spend = clean.loc[
    clean["type_clean"] == "debit", "amount"
].sum()


# ------------------------------------------------------------
# CATEGORY SPENDING
# ------------------------------------------------------------

category_spend = (
    clean[clean["type_clean"] == "debit"]
    .groupby("category")["amount"]
    .sum()
)


# ------------------------------------------------------------
# CATEGORY PERCENTAGES
# ------------------------------------------------------------

food_spend = (
    category_spend.get("Food Delivery", 0)
    + category_spend.get("Restaurants", 0)
    + category_spend.get("Cafe", 0)
)

food_percent = (
    food_spend / total_spend * 100
    if total_spend != 0 else 0
)

quick_percent = (
    category_spend.get("Quick Commerce", 0)
    / total_spend * 100
    if total_spend != 0 else 0
)

ecommerce_percent = (
    category_spend.get("E-commerce", 0)
    / total_spend * 100
    if total_spend != 0 else 0
)

investment_percent = (
    category_spend.get("Investments", 0)
    / total_spend * 100
    if total_spend != 0 else 0
)

transport_percent = (
    category_spend.get("Transport", 0)
    / total_spend * 100
    if total_spend != 0 else 0
)


# ------------------------------------------------------------
# LATE-NIGHT FOOD DELIVERY
# ------------------------------------------------------------

food_delivery = clean[
    (clean["type_clean"] == "debit") &
    (clean["category"] == "Food Delivery")
].copy()

food_delivery["hour"] = (
    food_delivery["Time"]
    .astype(str)
    .str[:2]
    .astype(int)
)

late_night_food = food_delivery[
    (food_delivery["hour"] >= 21) |
    (food_delivery["hour"] <= 2)
]

late_night_percent = (
    len(late_night_food) / len(food_delivery) * 100
    if len(food_delivery) != 0 else 0
)


# ------------------------------------------------------------
# SUBSCRIPTION VENDORS
# ------------------------------------------------------------

subscription_vendors = clean[
    clean["category"] == "Subscriptions"
]["vendor_clean"].nunique()


# ------------------------------------------------------------
# SAVINGS RATE
# ------------------------------------------------------------

savings_rate_archetype = (
    (total_credits - total_debits)
    / total_credits * 100
    if total_credits != 0 else 0
)
print("ARCHETYPE METRICS")
print("=" * 70)

print(f"Food spending       : {food_percent:.2f}%")
print(f"Quick Commerce      : {quick_percent:.2f}%")
print(f"E-commerce          : {ecommerce_percent:.2f}%")
print(f"Investments         : {investment_percent:.2f}%")
print(f"Transport           : {transport_percent:.2f}%")
print(f"Late-night food     : {late_night_percent:.2f}%")
print(f"Subscription vendors: {subscription_vendors}")
print(f"Savings rate        : {savings_rate_archetype:.2f}%")

ARCHETYPE METRICS
Food spending       : 17.60%
Quick Commerce      : 8.21%
E-commerce          : 35.97%
Investments         : 14.78%
Transport           : 3.42%
Late-night food     : 20.90%
Subscription vendors: 3
Savings rate        : -229.34%


Apply All 8 Archetype Rules

In [49]:
archetypes = []


# ------------------------------------------------------------
# 1. THE FOODIE
# ------------------------------------------------------------

if food_percent > 25:
    archetypes.append(
        (
            "THE FOODIE",
            f"{food_percent:.1f}% of spending on food"
        )
    )


# ------------------------------------------------------------
# 2. THE QUICK COMMERCE JUNKIE
# ------------------------------------------------------------

if quick_percent > 15:
    archetypes.append(
        (
            "THE QUICK COMMERCE JUNKIE",
            f"{quick_percent:.1f}% of spending on Quick Commerce"
        )
    )


# ------------------------------------------------------------
# 3. THE SHOPAHOLIC
# ------------------------------------------------------------

if ecommerce_percent > 15:
    archetypes.append(
        (
            "THE SHOPAHOLIC",
            f"{ecommerce_percent:.1f}% of spending on E-commerce"
        )
    )


# ------------------------------------------------------------
# 4. THE INVESTOR
# ------------------------------------------------------------

if investment_percent > 15:
    archetypes.append(
        (
            "THE INVESTOR",
            f"{investment_percent:.1f}% of spending on Investments"
        )
    )


# ------------------------------------------------------------
# 5. THE LATE-NIGHT SNACKER
# ------------------------------------------------------------

if late_night_percent > 50:
    archetypes.append(
        (
            "THE LATE-NIGHT SNACKER",
            f"{late_night_percent:.1f}% of Food Delivery transactions late at night"
        )
    )


# ------------------------------------------------------------
# 6. THE CAB COMMUTER
# ------------------------------------------------------------

if transport_percent > 10:
    archetypes.append(
        (
            "THE CAB COMMUTER",
            f"{transport_percent:.1f}% of spending on Transport"
        )
    )


# ------------------------------------------------------------
# 7. THE SUBSCRIPTION LOVER
# ------------------------------------------------------------

if subscription_vendors >= 5:
    archetypes.append(
        (
            "THE SUBSCRIPTION LOVER",
            f"{subscription_vendors} subscription vendors"
        )
    )


# ------------------------------------------------------------
# 8. THE YOLO SPENDER
# ------------------------------------------------------------

if savings_rate_archetype < 10:
    archetypes.append(
        (
            "THE YOLO SPENDER",
            f"Savings rate = {savings_rate_archetype:.1f}%"
        )
    )


# ------------------------------------------------------------
# 9. THE DISCIPLINED SAVER
# ------------------------------------------------------------

if savings_rate_archetype > 40:
    archetypes.append(
        (
            "THE DISCIPLINED SAVER",
            f"Savings rate = {savings_rate_archetype:.1f}%"
        )
    )


print("Archetype rules applied successfully.")
print("Archetypes detected:", len(archetypes))

Archetype rules applied successfully.
Archetypes detected: 2


Final Archetype Report

In [50]:

print("=" * 75)
print("                 SPENDING ARCHETYPES")
print("=" * 75)

if len(archetypes) == 0:

    print("\nNo archetypes matched the defined rules.")

else:

    print("\nDetected archetypes:\n")

    for name, metric in archetypes:

        print(f"-> {name}")
        print(f"   Supporting metric: {metric}")
        print()


print("=" * 75)
print(f"TOTAL ARCHETYPES DETECTED: {len(archetypes)}")
print("=" * 75)

print("\nFEATURE 8 STATUS: PASSED")

                 SPENDING ARCHETYPES

Detected archetypes:

-> THE SHOPAHOLIC
   Supporting metric: 36.0% of spending on E-commerce

-> THE YOLO SPENDER
   Supporting metric: Savings rate = -229.3%

TOTAL ARCHETYPES DETECTED: 2

FEATURE 8 STATUS: PASSED


Final Report

This is the  final report consolidates the results obtained from all major

SpendDNA
analysis features into one executive-level summary.

It presents the overall financial position, top spending categories,
top vendors, monthly spending trends, time-of-day patterns, spending
anomalies, and identified spending archetypes.

The report is generated from the cleaned transaction dataset and uses
the calculated results from the previous analysis features. This makes
the final output easy to interpret and provides a concise overview of
the user's spending behavior.

The report is designed to be mentor-friendly by using clear section
headings, aligned values, percentages, transaction counts, and
human-readable financial amounts.

In [52]:
# ============================================================
# SPENDDNA - FINAL EXECUTIVE REPORT
# ============================================================
print()
print("=" * 80)
print("                         SPENDDNA REPORT")
print("                    TRANSACTION ANALYSIS")
print("=" * 80)

print()
print("EXECUTIVE SUMMARY")
print("-" * 80)

print(f"Total credits    : Rs. {total_credits:,.2f}")
print(f"Total debits     : Rs. {total_debits:,.2f}")
print(f"Net savings      : Rs. {total_credits - total_debits:,.2f}")
print(f"Savings rate     : {savings_rate_archetype:.1f}%")
print(f"Transactions     : {len(clean)}")
print(f"Unique vendors   : {clean['vendor_clean'].nunique()}")


# ============================================================
# TOP CATEGORIES
# ============================================================

print()
print("TOP CATEGORIES (% OF DEBIT TOTAL)")
print("-" * 80)

category_summary = (
    clean[clean["type_clean"] == "debit"]
    .groupby("category")["amount"]
    .sum()
    .sort_values(ascending=False)
)

for category, amount in category_summary.head(5).items():

    percentage = (
        amount / total_debits * 100
        if total_debits != 0 else 0
    )

    bars = "#" * int(percentage / 2)

    print(
        f"{category:<20} "
        f"{bars:<20} "
        f"{percentage:>5.1f}% "
        f"Rs. {amount:>12,.2f}"
    )


# ============================================================
# TOP VENDORS
# ============================================================

print()
print("TOP VENDORS")
print("-" * 80)

vendor_summary = (
    clean[clean["type_clean"] == "debit"]
    .groupby("vendor_clean")
    .agg(
        spend=("amount", "sum"),
        transactions=("amount", "count")
    )
    .sort_values("spend", ascending=False)
)

for vendor, row in vendor_summary.head(5).iterrows():

    print(
        f"{vendor:<20} "
        f"Rs. {row['spend']:>12,.2f} "
        f"({int(row['transactions'])} transactions)"
    )


# ============================================================
# TIME-OF-DAY
# ============================================================

print()
print("TIME-OF-DAY PATTERNS")
print("-" * 80)

print(
    f"Overall peak spending hour : "
    f"{peak_hour:02d}:00 "
    f"(Rs. {peak_amount:,.2f})"
)

print()

for category in category_peak_hours.index:

    hour = category_peak_hours[category]
    amount = category_peak_amounts[category]

    print(
        f"{category:<20} "
        f"{hour:02d}:00 "
        f"Rs. {amount:,.2f}"
    )


# ============================================================
# MONTHLY TREND
# ============================================================

print()
print("MONTHLY SPENDING TREND")
print("-" * 80)

print(month_pivot.round(2).to_string())

print()

print(
    f"Biggest growth  : "
    f"{biggest_growth_category} "
    f"({biggest_growth_percent:.1f}%)"
)

print(
    f"Biggest decline : "
    f"{biggest_decline_category} "
    f"({biggest_decline_percent:.1f}%)"
)


# ============================================================
# ANOMALIES
# ============================================================

print()
print("TOP 5 SPENDING ANOMALIES")
print("-" * 80)

for _, row in anomalies.head(5).iterrows():

    print(
        f"{str(row['Date']):<14} "
        f"{row['vendor_clean']:<15} "
        f"{row['category']:<20} "
        f"Rs. {row['amount']:>10,.2f} "
        f"(z={row['z_score']:.2f})"
    )

print()
print(f"Total anomalies detected : {len(anomalies)}")


# ============================================================
# SPENDING ARCHETYPES
# ============================================================

print()
print("SPENDING ARCHETYPES")
print("-" * 80)

if len(archetypes) == 0:

    print("No predefined archetype rules were matched.")

else:

    for name, metric in archetypes:

        print(f"-> {name}")
        print(f"   {metric}")


# ============================================================
# FINAL STATUS
# ============================================================

print()
print("=" * 80)
print("                    ANALYSIS COMPLETE")
print("=" * 80)


                         SPENDDNA REPORT
                    TRANSACTION ANALYSIS

EXECUTIVE SUMMARY
--------------------------------------------------------------------------------
Total credits    : Rs. 509,774.00
Total debits     : Rs. 1,678,901.00
Net savings      : Rs. -1,169,127.00
Savings rate     : -229.3%
Transactions     : 1310
Unique vendors   : 38

TOP CATEGORIES (% OF DEBIT TOTAL)
--------------------------------------------------------------------------------
E-commerce           #################     36.0% Rs.   603,877.00
Investments          #######               14.8% Rs.   248,160.00
Food Delivery        ####                   8.7% Rs.   146,249.00
Quick Commerce       ####                   8.2% Rs.   137,879.00
Personal Transfer    ####                   8.0% Rs.   134,331.00

TOP VENDORS
--------------------------------------------------------------------------------
Amazon               Rs.   328,530.00 (86 transactions)
Zerodha              Rs.   210,000.00 (14

## Key Insights

1. **E-commerce is the largest spending category.** It accounts for
36.0% of total debit spending, making online shopping the biggest
contributor to overall expenditure.

2. **Some transactions are unusually high compared with their
category spending patterns.** The anomaly analysis detected 37
anomalous transactions, representing 2.84% of debit transactions.
The highest anomaly was a Rs. 7,441 Cafe transaction with a z-score
of 9.61.

3. **The spending behavior indicates overspending.** The analysis
identified two spending archetypes: The Shopaholic and The YOLO
Spender. The Shopaholic classification is supported by 36.0%
E-commerce spending, while the YOLO Spender classification is
supported by the -229.3% savings rate.

## Reflection

This project helped me understand the complete process of transaction
data analysis, starting from raw-data inspection and cleaning and
continuing through vendor extraction, category classification,
spending trends, time-of-day analysis, anomaly detection, and
spending archetype identification.

I learned that data cleaning and standardization are important because
the quality of the input data directly affects the accuracy of the
analysis. I also gained practical experience using Pandas and
statistical methods such as z-scores to convert transaction data into
meaningful financial insights.

Overall, this project improved my ability to perform end-to-end data
analysis and present the results in a clear and understandable way.

## AI Assistance Disclosure

AI assistance was used during this project for code guidance,
debugging, explanation of concepts, and formatting suggestions.

I reviewed, executed, and validated the analysis and final outputs
using the provided transaction dataset.